## 欢迎回到 Python Notebooks！

想我了吗？？

### 欢迎来到第四周第二天——LangGraph 入门！

In [ ]:
# ============================================================
#  必须在所有 import 之前：禁用系统代理对本机地址的影响
#  这台机器有 127.0.0.1:7897 代理，httpx 会读它导致 Gradio 自检 502
# ============================================================
import os
os.environ["HTTP_PROXY"] = ""
os.environ["HTTPS_PROXY"] = ""
os.environ["NO_PROXY"] = "127.0.0.1,localhost"

from typing import Annotated
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
import random

# 加载环境变量（从 .env 读取 DEEPSEEK_API_KEY 等）
load_dotenv(override=True)

In [ ]:
# 一些有用的常量
# 名词列表
nouns = ["Cabbages", "Unicorns", "Toasters", "Penguins", "Bananas", "Zombies", "Rainbows", "Eels", "Pickles", "Muffins"]
# 形容词列表
adjectives = ["outrageous", "smelly", "pedantic", "existential", "moody", "sparkly", "untrustworthy", "sarcastic", "squishy", "haunted"]

In [ ]:
# 我们最熟悉的第一步！顺便说一下，Crew 之前一直在帮我们做这个。
load_dotenv(override=True) #强制用 .env 文件中的值覆盖已有的同名环境变量

In [ ]:
# def shout(text: str) -> str:  这是普通类型函数
# def shout(text: Annotated[str, "something to be shouted"]) -> str:
# 除了参数还附带额外说明的函数，和普通类型执行起来没有区别，但在一些工具中会有特殊的处理，比如在 LangSmith 中会把 Annotated 的说明作为参数的描述展示出来。
def shout(text: Annotated[str, "something to be shouted"]) -> str:
    print(text.upper())
    return text.upper()

shout("hello")

### 关于 "Annotated" 的一点说明

你可能已经知道：类型提示是 Python 的一个特性，可以指定某个变量的类型：

`my_favorite_things: List`

但你可能不知道：

你还可以使用一种叫做 "Annotated" 的方式来添加额外的信息，供其他人使用：

`my_favorite_things: Annotated[List, "these are a few of mine"]`

LangGraph 要求我们在定义 State 对象时使用这个特性。

它需要我们告诉它：**应该调用什么函数来用新值更新 State**。

这个函数叫做 **reducer（合并器）**。

LangGraph 提供了一个默认的 reducer 叫做 `add_messages`，它可以处理最常见的情况。

这应该能解释为什么 State 看起来是这个样子。

### 第 1 步：定义 State 对象

你可以使用任何 Python 对象；但最常见的是使用 TypedDict 或 Pydantic BaseModel。

In [ ]:
# 定义一个结构体  add_messages是 LangGraph 提供了默认的 reducer
# reducer 通常指的是 reduce() 函数。
# eg f(固定函数,接收n个参数)。给定一个范围参数。
# reducer将范围内的元素依次传入固定函数进行处理，最终得到一个结果。
# add_messages保证了在对话中每一步的输入输出都会被记录下来，
# 形成一个完整的对话历史。这对于调试和分析非常有用。没有就不会有完整的对话记录了。
class State(BaseModel):
        
    messages: Annotated[list, add_messages]


### 第 2 步：用这个 State 类启动 Graph Builder

In [ ]:
graph_builder = StateGraph(State)

# Go 与 Python 基础数据类型对比

## 🔢 数字类型
| 类别   | Go 类型                                | Python 类型                          |
|--------|----------------------------------------|--------------------------------------|
| 整数   | int, int8/16/32/64, uint8~64           | int（自动扩展，无溢出）              |
| 浮点数 | float32, float64                       | float（双精度 IEEE 754）             |
| 复数   | complex64, complex128                  | complex (如 3+4j)                    |
| 布尔   | bool (true/false)                      | bool (True/False，int 子类)          |

---

## 📝 字符类型
| 类别   | Go 类型                                | Python 类型                          |
|--------|----------------------------------------|--------------------------------------|
| 字符   | rune (int32, Unicode), byte (uint8)    | str 单字符 (长度为 1 的字符串)       |
| 字符串 | string (UTF-8，不可变)                 | str (Unicode，不可变)                |
| 字节串 | byte 数组                              | bytes (字节串)                       |

---

## 📦 容器类型
| 类别   | Go 类型                                | Python 类型                          |
|--------|----------------------------------------|--------------------------------------|
| 数组   | array (固定长度)                       | list (动态长度，可变)                |
| 序列   | slice (灵活引用数组)                   | tuple (不可变序列)                   |
| 集合   | map (键值对)                           | dict (键值对)                        |
| 集合   | 无原生 set 类型                        | set (集合)                           |
| 结构体 | struct (复合类型)                      | class (面向对象，支持继承)           |
| 并发   | channel (并发通信)                     | 无直接等价，用 queue/asyncio         |


### 第 3 步：创建一个 Node

一个 node 可以是任意 Python 函数。

我们之前设置的 reducer 会被自动调用，用来将当前响应与之前的响应合并。

In [ ]:
def our_first_node(old_state: State) -> State:
    #f"" 将其中内容转字符串，
    # {random.choice(nouns)} 从名词列表中随机选一个，
    # {random.choice(adjectives)} 从形容词列表中随机选一个
    reply = f"{random.choice(nouns)} are {random.choice(adjectives)}"
    # 这边是设置agent的角色和风格
    messages = [{"role": "assistant", "content": reply}]
    print(messages)
    new_state = State(messages=messages)

    return new_state

graph_builder.add_node("first_node", our_first_node)

### 第 4 步：创建 Edges（边）

In [ ]:
graph_builder.add_edge(START, "first_node")
graph_builder.add_edge("first_node", END)

### 第 5 步：编译 Graph

In [ ]:
graph = graph_builder.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

### 就这样！开始展示！

In [ ]:
def chat(user_input: str, history):
    message = {"role": "user", "content": user_input}
    messages = [message]
    state = State(messages=messages)
    result = graph.invoke(state)
    print(result)
    return result["messages"][-1].content


gr.ChatInterface(chat, type="messages").launch()

In [ ]:
gr.close_all()

### 为什么我要先展示这个？

为了说明一个重点：**LangGraph 的核心是 Python 函数——它不一定需要涉及 LLM！！**

现在我们再来一遍这 5 个步骤，但一气呵成：

In [ ]:
# StateGraph 是 LangGraph 的核心类.就是一个工作流引擎——
# 它让你把多个处理步骤（Python 函数）串成一个有向图，
# 自动管理状态在节点之间的传递和合并（通过 reducer 如 add_messages）。有5步组成
# 第 1 步：定义 State 对象
class State(BaseModel):
    messages: Annotated[list, add_messages]

In [ ]:
# 第 2 步：用这个 State 类启动 Graph Builder
graph_builder = StateGraph(State)

In [ ]:
# 第 3 步：创建一个 Node

# 使用 DeepSeek 大模型（通过 OpenAI 兼容接口）
llm = ChatOpenAI(
    model=os.getenv("DEEPSEEK_MODEL", "deepseek-v4-pro"),
    base_url=os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
)

def chatbot_node(old_state: State) -> State:
    response = llm.invoke(old_state.messages)
    new_state = State(messages=[response])
    return new_state

graph_builder.add_node("chatbot", chatbot_node)

In [ ]:
# 第 4 步：创建 Edges（边）
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

In [ ]:
# 第 5 步：编译 Graph
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

### 搞定！接下来，让我们做这个：

In [ ]:
def chat(user_input: str, history):
    initial_state = State(messages=[{"role": "user", "content": user_input}])
    result = graph.invoke(initial_state)
    print(result)
    #只返回模型回答的内容
    return result['messages'][-1].content

gr.ChatInterface(chat, type="messages").launch()

In [ ]:
gr.close_all()